# Diffusion Models

Suppose you want to generate new handwritten digits that look like they came from the MNIST dataset. You could train a GAN and pit a generator against a discriminator — but adversarial training is notoriously unstable. You could train a VAE, but its generated samples tend to be blurry. **Diffusion models** [@ddpm] offer a third way: learn to reverse a gradual noising process. During the **forward process**, we progressively corrupt a data sample $\mathbf{x}_0$ over $T$ steps by adding small amounts of Gaussian noise at each step, until the signal is almost entirely destroyed and the result is approximately $\mathcal{N}(\mathbf{0}, \mathbf{I})$. The model learns the **reverse process**: starting from pure noise $\mathbf{x}_T$, iteratively denoise over $T$ steps to recover a sample from the data distribution.

Ho et al. [@ddpm] demonstrated that this framework, called **Denoising Diffusion Probabilistic Models** (DDPM), produces sharp, high-quality samples for image generation and became the foundation of modern systems like Stable Diffusion and DALL·E 2. Unlike GANs, there is no adversarial training instability; unlike VAEs, the generative process is not constrained by a learned encoder. The price is sampling speed: generating a single image requires $T = 1000$ neural network forward passes. **DDIM** [@ddim] later showed that a non-Markovian reformulation allows high-quality sampling with as few as 50 steps.

We implement DDPM from scratch on MNIST. We derive the forward and reverse processes, compare linear and cosine noise schedules, build and train a small U-Net denoiser, and implement both DDPM and DDIM samplers. The goal is a complete, self-contained implementation where every design choice can be traced back to the underlying math.

<br>

In [ ]:
import math
import torch
import warnings
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib_inline import backend_inline
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATASET_DIR = Path("./data").absolute()
RANDOM_SEED = 42
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.simplefilter(action="ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("device:", DEVICE)

## Data

**Data.** We train on MNIST, normalizing pixel values from $[0, 1]$ to $[-1, 1]$ via the transform $x \mapsto 2x - 1.$ This is standard practice for diffusion models: the forward process adds Gaussian noise with zero mean, so centering the data around zero keeps the signal and noise in comparable ranges throughout the noising trajectory.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # [0,1] -> [-1, 1]
])

train_dataset = datasets.MNIST(
    root=DATASET_DIR, train=True, download=True, transform=transform
)
train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True
)

print(f"Training samples : {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")

## Forward process

The forward process defines a Markov chain that gradually corrupts a data point $\mathbf{x}_0$ over $T$ discrete time steps. At each step $t$, we add a small amount of Gaussian noise controlled by a **variance schedule** $\beta_t \in (0, 1)$:

$$
q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{1 - \beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I}\right).
$$

The factor $\sqrt{1 - \beta_t}$ slightly shrinks the mean at each step, preventing the variance from exploding. To see what happens over many steps, define $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s.$ By induction, using the fact that a Gaussian times a Gaussian is a Gaussian (the **reproducing property**), we can sample $\mathbf{x}_t$ from $\mathbf{x}_0$ in a single step:

$$
\boxed{q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1 - \bar{\alpha}_t)\mathbf{I}\right).}
$$

**Derivation.** Suppose $q(\mathbf{x}_{t-1} | \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar\alpha_{t-1}}\,\mathbf{x}_0, (1-\bar\alpha_{t-1})\mathbf{I})$ (inductive hypothesis). We can write $\mathbf{x}_{t-1} = \sqrt{\bar\alpha_{t-1}}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_{t-1}}\,\boldsymbol{\epsilon}_{t-1}$ for $\boldsymbol{\epsilon}_{t-1} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).$ Substituting into the one-step transition:

$$
\begin{aligned}
\mathbf{x}_t
&= \sqrt{\alpha_t}\,\mathbf{x}_{t-1} + \sqrt{\beta_t}\,\boldsymbol{\epsilon}_t \\[0.75em]
&= \sqrt{\alpha_t}\left(\sqrt{\bar\alpha_{t-1}}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_{t-1}}\,\boldsymbol{\epsilon}_{t-1}\right) + \sqrt{\beta_t}\,\boldsymbol{\epsilon}_t \\[0.75em]
&= \sqrt{\alpha_t \bar\alpha_{t-1}}\,\mathbf{x}_0 + \underbrace{\sqrt{\alpha_t(1-\bar\alpha_{t-1})}\,\boldsymbol{\epsilon}_{t-1} + \sqrt{\beta_t}\,\boldsymbol{\epsilon}_t}_{\sim\,\mathcal{N}(\mathbf{0},\,(\alpha_t(1-\bar\alpha_{t-1})+\beta_t)\mathbf{I})} \\[0.75em]
&= \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol{\epsilon},
\end{aligned}
$$

since $\alpha_t(1-\bar\alpha_{t-1}) + \beta_t = \alpha_t - \alpha_t\bar\alpha_{t-1} + 1 - \alpha_t = 1 - \bar\alpha_t.$ The base case $t=1$ is immediate. Equivalently, this gives us the **noise addition formula**:

$$
\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).
$$

At each step $t$, the signal $\mathbf{x}_0$ is scaled down by $\sqrt{\bar{\alpha}_t}$ while noise is scaled up by $\sqrt{1 - \bar{\alpha}_t}.$ As $t \to T$, we have $\bar{\alpha}_T \approx 0$, so $\mathbf{x}_T \approx \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).$ The schedule $\{\beta_t\}$ controls how quickly the signal is destroyed.

## Noise schedules

The **linear schedule** from [@ddpm] sets $\beta_t$ to increase linearly from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$ over $T = 1000$ steps. We precompute and cache all the derived quantities needed for sampling:

In [ ]:
T = 1000

betas = torch.linspace(1e-4, 0.02, T)                              # <1>
alphas = 1.0 - betas                                                # <2>
alphas_cumprod = torch.cumprod(alphas, dim=0)                       # <3>
sqrt_alphas_cumprod = alphas_cumprod.sqrt()                         # <4>
sqrt_one_minus_alphas_cumprod = (1.0 - alphas_cumprod).sqrt()       # <5>

print(f"alpha_bar at t=  1: {alphas_cumprod[0]:.4f}")
print(f"alpha_bar at t=500: {alphas_cumprod[499]:.4f}")
print(f"alpha_bar at t=999: {alphas_cumprod[-1]:.6f}")

1. Linear schedule: $\beta_t$ from $10^{-4}$ to $0.02$ uniformly over $T$ steps.
2. $\alpha_t = 1 - \beta_t$; signal retention coefficient at each step.
3. $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$; cumulative signal retention.
4. $\sqrt{\bar{\alpha}_t}$; signal scale in the one-shot noise formula.
5. $\sqrt{1 - \bar{\alpha}_t}$; noise scale in the one-shot formula.

Plotting the signal retention $\bar{\alpha}_t$ over the diffusion trajectory:

In [ ]:
#| code-fold: true
ts = np.arange(1, T + 1)

plt.figure(figsize=(6, 3))
plt.plot(ts, alphas_cumprod.numpy(), color="C0", linewidth=2, label=r"$\bar{\alpha}_t$ (linear)")
plt.xlabel("$t$")
plt.ylabel(r"$\bar{\alpha}_t$")
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show();

**Figure.** Signal-to-noise decay under the linear schedule. By step $t = 500$, about $6\%$ of the signal remains; by $t = 999$, it is essentially zero. The schedule is designed so that $\mathbf{x}_T$ is indistinguishable from $\mathcal{N}(\mathbf{0}, \mathbf{I}).$

<br>

We can now visualize the forward process directly. We apply the one-shot formula $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\epsilon}$ to show how a single MNIST digit is progressively corrupted:

In [ ]:
#| code-fold: true
def q_sample_simple(x0, t, sqrt_acp, sqrt_one_minus_acp):
    """One-shot forward sample: x_t = sqrt(abar_t)*x0 + sqrt(1-abar_t)*eps."""
    noise = torch.randn_like(x0)
    return sqrt_acp[t] * x0 + sqrt_one_minus_acp[t] * noise


# Grab a single digit
x0, label = train_dataset[7]
x0 = x0.unsqueeze(0)   # (1, 1, 28, 28)

timesteps_to_show = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(14, 1.8))
for ax, t in zip(axes, timesteps_to_show):
    if t == 0:
        xt = x0
    else:
        xt = q_sample_simple(x0, t - 1, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod)
    img = xt.squeeze().numpy()
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"$t={t}$", fontsize=8)
    ax.axis("off")

fig.tight_layout()
plt.show();

**Figure.** Forward process applied to a single MNIST digit. The image structure is visible up to around $t = 400$ and is completely destroyed by $t = 999.$ The reverse process must learn to undo this trajectory.

## Reverse process and training objective

The reverse process models $p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t) = \mathcal{N}(\boldsymbol{\mu}_\theta(\mathbf{x}_t, t),\, \sigma_t^2 \mathbf{I})$: given the noisy image at step $t$, predict the less-noisy image at step $t-1.$ To derive a tractable training objective, we note that the **posterior** $q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0)$ is Gaussian (because the forward process is Gaussian), and we compute it by Bayes' rule:

$$
q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) \propto q(\mathbf{x}_t | \mathbf{x}_{t-1})\, q(\mathbf{x}_{t-1} | \mathbf{x}_0).
$$

Completing the square in the exponent gives:

$$
q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}\!\left(\tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0),\; \tilde{\beta}_t \mathbf{I}\right)
$$

where the posterior mean and variance are:

$$
\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1 - \bar{\alpha}_t}\,\mathbf{x}_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t}\,\mathbf{x}_t,
\qquad
\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\,\beta_t.
$$

Now we substitute the noise addition formula $\mathbf{x}_0 = (\mathbf{x}_t - \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon})/\sqrt{\bar{\alpha}_t}$ into $\tilde{\boldsymbol{\mu}}_t$ to re-express the posterior mean purely in terms of $\mathbf{x}_t$ and the noise $\boldsymbol{\epsilon}$:

$$
\tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \boldsymbol{\epsilon}) = \frac{1}{\sqrt{\alpha_t}}\!\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\boldsymbol{\epsilon}\right).
$$

This suggests training a neural network $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ to predict the noise $\boldsymbol{\epsilon}$ from the noisy image $\mathbf{x}_t$ at time $t.$ The variational lower bound on the log-likelihood reduces to comparing the predicted and true posterior means, but Ho et al. [@ddpm] found that dropping the time-dependent weighting coefficients and using the simplified objective works better in practice:

$$
\boxed{L_\text{simple} = \mathbb{E}_{t,\, \mathbf{x}_0,\, \boldsymbol{\epsilon}}\!\left[\left\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta\!\left(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon},\; t\right)\right\|^2\right].}
$$

This is a **noise prediction** objective: at each training step we (1) sample a clean image $\mathbf{x}_0$, (2) sample a timestep $t$ uniformly, (3) sample noise $\boldsymbol{\epsilon}$, (4) corrupt the image to get $\mathbf{x}_t$, and (5) ask the model to predict the noise that was added. At inference time, the predicted noise is used to recover the posterior mean $\tilde{\boldsymbol{\mu}}_t$, and we step the denoising chain.

:::{.callout-note}
The simplified objective $L_\text{simple}$ is equivalent to a weighted mixture of denoising score matching objectives at different noise levels. Each timestep $t$ corresponds to a different signal-to-noise ratio, and the uniform sampling over $t$ ensures the model learns to denoise across the full trajectory.

:::

## U-Net architecture

The denoiser $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ takes a noisy image $\mathbf{x}_t \in \mathbb{R}^{1 \times 28 \times 28}$ and a scalar timestep $t \in \{1, \ldots, T\}$ as inputs and outputs a noise estimate of the same shape. We use a **U-Net** [@unet]: an encoder path that progressively downsamples the spatial resolution while increasing channel depth, and a symmetric decoder path that upsamples back to the original resolution. **Skip connections** concatenate encoder feature maps to the corresponding decoder features, preserving fine spatial structure that would otherwise be lost during downsampling.

The timestep $t$ is embedded with **sinusoidal positional encoding** — the same idea used in Transformers ([NB09](../09-attention-transformers.html)) — and injected into each residual block via addition. This gives the model a continuous sense of "how noisy" the input is, which is essential because the denoising operation differs dramatically between early ($t \approx T$, mostly noise) and late ($t \approx 1$, mostly signal) steps.

**Architecture.** For $28 \times 28$ MNIST images, we use three resolution levels: $28 \times 28 \to 14 \times 14 \to 7 \times 7$ in the encoder, and the reverse in the decoder. At each resolution, we apply two residual blocks. The bottleneck at $7 \times 7$ has the most channels. Channel widths are $[32, 64, 128]$ for the encoder and mirror in the decoder, with the base expanded by skip-connection concatenation. Downsampling uses stride-$2$ convolutions; upsampling uses `ConvTranspose2d`. Each residual block uses GroupNorm and SiLU activations.

Defining the sinusoidal timestep embedding:

In [ ]:
class SinusoidalEmbedding(nn.Module):                              # <1>
    """Sinusoidal embedding of a scalar timestep t."""

    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:            # <2>
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        )                                                          # <3>
        args = t[:, None].float() * freqs[None, :]                # <4>
        emb = torch.cat([args.sin(), args.cos()], dim=-1)          # <5>
        return emb

1. The sinusoidal embedding maps a scalar timestep $t$ to a vector in $\mathbb{R}^{\text{dim}}$ using sine and cosine at geometrically-spaced frequencies, following [@transformers]. The same encoding is used for positional encoding in [NB09](../09-attention-transformers.html).
2. `t` has shape `(B,)` — a batch of integer timesteps.
3. Frequencies $\omega_k = 10000^{-k/(d/2-1)}$ for $k = 0, \ldots, d/2 - 1$; geometrically spaced from $1$ down to $10^{-4}$.
4. Outer product: shape `(B, d/2)` where each row is $t \cdot \omega_k$ for all $k$.
5. Concatenate sine and cosine halves to produce `(B, dim)`.

Defining the residual block with timestep conditioning:

In [ ]:
class ResBlock(nn.Module):
    """Conv block with timestep conditioning and residual skip connection."""

    def __init__(self, in_ch: int, out_ch: int, time_emb_dim: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)                       # <1>
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)            # <2>
        self.skip = (
            nn.Conv2d(in_ch, out_ch, 1)
            if in_ch != out_ch else nn.Identity()                  # <3>
        )
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        # x: (B, in_ch, H, W)   t_emb: (B, time_emb_dim)
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.time_mlp(self.act(t_emb))[:, :, None, None]  # <4>
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.skip(x)                                    # <5>

1. GroupNorm with 8 groups is preferred over BatchNorm in diffusion U-Nets since batch statistics are unreliable when the batch contains images at many different noise levels.
2. The time MLP projects the time embedding to `out_ch` channels so it can be added to the spatial feature map.
3. When channel counts change, we use a $1 \times 1$ convolution to match dimensions in the residual path; otherwise the identity is used.
4. Broadcast-add the time embedding: `[:, :, None, None]` expands `(B, out_ch)` to `(B, out_ch, 1, 1)` for spatial addition.
5. Residual connection: adding the (projected) input allows gradients to flow directly through the block.

Assembling the full U-Net:

In [ ]:
class UNet(nn.Module):
    """Small U-Net denoiser for 28x28 single-channel images."""

    def __init__(self, in_ch: int = 1, base_ch: int = 32, time_emb_dim: int = 128):
        super().__init__()

        # Timestep embedding
        self.time_emb = nn.Sequential(
            SinusoidalEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        # Encoder
        ch = [base_ch, base_ch * 2, base_ch * 4]                  # [32, 64, 128]
        self.enc0 = ResBlock(in_ch,  ch[0], time_emb_dim)          # (B,  1, 28,28) -> (B,32,28,28)
        self.down0 = nn.Conv2d(ch[0], ch[0], 4, stride=2, padding=1)  # -> (B,32,14,14)
        self.enc1 = ResBlock(ch[0], ch[1], time_emb_dim)           # -> (B,64,14,14)
        self.down1 = nn.Conv2d(ch[1], ch[1], 4, stride=2, padding=1)  # -> (B,64, 7, 7)

        # Bottleneck
        self.bot0 = ResBlock(ch[1], ch[2], time_emb_dim)           # -> (B,128,7, 7)
        self.bot1 = ResBlock(ch[2], ch[2], time_emb_dim)           # -> (B,128,7, 7)

        # Decoder
        self.up1 = nn.ConvTranspose2d(ch[2], ch[1], 4, stride=2, padding=1)  # -> (B,64,14,14)
        self.dec1 = ResBlock(ch[1] + ch[1], ch[1], time_emb_dim)   # skip concat: 64+64 -> (B,64,14,14)
        self.up0 = nn.ConvTranspose2d(ch[1], ch[0], 4, stride=2, padding=1)  # -> (B,32,28,28)
        self.dec0 = ResBlock(ch[0] + ch[0], ch[0], time_emb_dim)   # skip concat: 32+32 -> (B,32,28,28)

        # Output head
        self.out = nn.Sequential(
            nn.GroupNorm(8, ch[0]),
            nn.SiLU(),
            nn.Conv2d(ch[0], in_ch, 1),                            # -> (B,1,28,28)
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # x: (B,1,28,28)  t: (B,) integer timesteps
        t_emb = self.time_emb(t)                                   # (B, time_emb_dim)

        # Encoder path
        e0 = self.enc0(x, t_emb)                                   # (B, 32, 28, 28)
        e1 = self.enc1(self.down0(e0), t_emb)                      # (B, 64, 14, 14)

        # Bottleneck
        b = self.bot0(self.down1(e1), t_emb)                       # (B,128,  7,  7)
        b = self.bot1(b, t_emb)                                     # (B,128,  7,  7)

        # Decoder path with skip connections
        d1 = self.dec1(torch.cat([self.up1(b), e1], dim=1), t_emb) # (B, 64, 14, 14)
        d0 = self.dec0(torch.cat([self.up0(d1), e0], dim=1), t_emb)# (B, 32, 28, 28)

        return self.out(d0)                                         # (B,  1, 28, 28)


# Parameter count
_model_tmp = UNet()
n_params = sum(p.numel() for p in _model_tmp.parameters())
print(f"U-Net parameters: {n_params:,}")
del _model_tmp

## Training

The DDPM training algorithm repeats the following at each step:

1. Sample a clean image $\mathbf{x}_0 \sim q(\mathbf{x}_0)$ from the training set.
2. Sample a timestep $t \sim \text{Uniform}\{1, \ldots, T\}$ uniformly at random.
3. Sample noise $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ of the same shape as $\mathbf{x}_0.$
4. Compute the noisy image $\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon}$ using the precomputed schedule.
5. Compute the predicted noise $\hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$.
6. Minimize $\|\boldsymbol{\epsilon} - \hat{\boldsymbol{\epsilon}}\|^2$ via gradient descent.

This uniform sampling over $t$ ensures that the model trains equally across all noise levels, which is important for the full reverse trajectory at inference time.

We encapsulate the noise schedule and the core sampling operations in a single class:

In [ ]:
class DiffusionSchedule:
    """Linear DDPM noise schedule with q_sample and p_mean helpers."""

    def __init__(
        self,
        T: int = 1000,
        beta_start: float = 1e-4,
        beta_end: float = 0.02,
        device: str = "cpu",
    ):
        self.T = T
        self.device = device

        betas = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas
        acp = torch.cumprod(alphas, dim=0)
        acp_prev = F.pad(acp[:-1], (1, 0), value=1.0)             # <1>

        self.register("betas", betas)
        self.register("alphas", alphas)
        self.register("acp", acp)                                  # alpha_bar_t
        self.register("acp_prev", acp_prev)                        # alpha_bar_{t-1}
        self.register("sqrt_acp", acp.sqrt())
        self.register("sqrt_one_minus_acp", (1.0 - acp).sqrt())
        self.register("posterior_var", betas * (1.0 - acp_prev) / (1.0 - acp))  # <2>

    def register(self, name: str, tensor: torch.Tensor):
        setattr(self, name, tensor)

    def q_sample(
        self,
        x0: torch.Tensor,
        t: torch.Tensor,
        noise: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward diffusion: sample x_t given x_0 and timestep t."""
        if noise is None:
            noise = torch.randn_like(x0)
        s = self.sqrt_acp[t][:, None, None, None]                  # <3>
        ns = self.sqrt_one_minus_acp[t][:, None, None, None]
        return s * x0 + ns * noise, noise

    def p_mean(
        self,
        model: nn.Module,
        xt: torch.Tensor,
        t: torch.Tensor,
    ) -> torch.Tensor:
        """Compute mu_theta(x_t, t) = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-abar_t) * eps_theta)."""
        eps_pred = model(xt, t)
        coef = self.betas[t] / self.sqrt_one_minus_acp[t]         # <4>
        coef = coef[:, None, None, None]
        mean = (xt - coef * eps_pred) / self.alphas[t][:, None, None, None]
        return mean

1. $\bar{\alpha}_{t-1}$ is needed for the posterior variance $\tilde{\beta}_t$; we prepend $\bar{\alpha}_0 = 1$ so the index aligns with $t = 0, 1, \ldots, T-1$.
2. Posterior variance $\tilde{\beta}_t = \beta_t (1 - \bar{\alpha}_{t-1}) / (1 - \bar{\alpha}_t)$; used when adding noise during sampling.
3. Reshape schedule coefficients to `(B, 1, 1, 1)` for broadcasting with image tensors of shape `(B, C, H, W)`.
4. The coefficient in front of the noise prediction: $\beta_t / \sqrt{1 - \bar{\alpha}_t}$.

Training the U-Net denoiser with AdamW:

In [ ]:
#| output: false
EPOCHS = 20
LR = 2e-4

schedule = DiffusionSchedule(T=T, device=DEVICE)
model = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for x0, _ in train_loader:
        x0 = x0.to(DEVICE)
        B = x0.shape[0]

        # Sample random timesteps uniformly in {0, ..., T-1}
        t = torch.randint(0, T, (B,), device=DEVICE)

        # Forward diffusion: corrupt x0 to x_t
        xt, noise = schedule.q_sample(x0, t)

        # Predict the noise
        noise_pred = model(xt, t)

        # L_simple: MSE between true and predicted noise
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    history.append(avg_loss)
    print(f"Epoch {epoch:>2}/{EPOCHS}  loss={avg_loss:.4f}")

Plotting the training loss:

In [ ]:
#| code-fold: true
plt.figure(figsize=(6, 3))
plt.plot(range(1, len(history) + 1), history, color="C0", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(linestyle="dotted", alpha=0.6)
plt.tight_layout()
plt.show();

## Sampling

Sampling runs the reverse process for $T$ steps. We start from $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and iteratively apply the reverse step. At each step $t$ from $T$ down to $1$, the model predicts $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ and we compute:

$$
\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\right) + \sigma_t \mathbf{z},
$$

where $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ for $t > 1$ and $\mathbf{z} = \mathbf{0}$ for $t = 1$ (no noise added at the final step), and $\sigma_t^2 = \tilde{\beta}_t = \beta_t (1 - \bar{\alpha}_{t-1}) / (1 - \bar{\alpha}_t)$ is the posterior variance.

:::{.callout-tip}
The final step $t = 1 \to 0$ sets $\mathbf{z} = \mathbf{0}$: we take the mean of the posterior rather than sampling from it. This avoids adding unnecessary noise when the image is already nearly clean.

:::

Implementing the DDPM reverse sampler:

In [ ]:
@torch.inference_mode()
def sample(
    model: nn.Module,
    schedule: DiffusionSchedule,
    n_samples: int,
    shape: tuple[int, ...],
    device: torch.device,
    return_trajectory: bool = False,
) -> torch.Tensor | list[torch.Tensor]:
    """DDPM reverse diffusion sampler. Returns samples in [-1, 1]."""
    model.eval()
    xt = torch.randn(n_samples, *shape, device=device)             # <1>
    trajectory = [xt.clone().cpu()] if return_trajectory else []

    for t_idx in reversed(range(schedule.T)):                      # <2>
        t_batch = torch.full((n_samples,), t_idx, device=device, dtype=torch.long)

        # Compute posterior mean
        mean = schedule.p_mean(model, xt, t_batch)                 # <3>

        # Add posterior noise (zero at final step)
        if t_idx > 0:
            sigma = schedule.posterior_var[t_idx].sqrt()
            xt = mean + sigma * torch.randn_like(xt)               # <4>
        else:
            xt = mean                                              # <5>

        if return_trajectory and t_idx in {799, 599, 399, 199, 0}:
            trajectory.append(xt.clone().cpu())

    return (xt.cpu(), trajectory) if return_trajectory else xt.cpu()

1. Start from pure Gaussian noise $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).$
2. Iterate $t = T-1, T-2, \ldots, 0$ (0-indexed; `t_idx=0` corresponds to the final step $t=1$ in the paper).
3. The posterior mean $\tilde{\boldsymbol{\mu}}_t$ expressed in terms of the predicted noise.
4. At intermediate steps, sample from the posterior by adding scaled Gaussian noise $\sigma_t \mathbf{z}$.
5. At the final step, return the mean directly to avoid adding noise to a near-clean image.

Generating a $5 \times 10$ grid of MNIST samples:

In [ ]:
#| code-fold: true
samples = sample(model, schedule, n_samples=50, shape=(1, 28, 28), device=DEVICE)
samples = samples.squeeze(1).numpy()  # (50, 28, 28)

fig, axes = plt.subplots(5, 10, figsize=(12, 6))
for ax, img in zip(axes.flat, samples):
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")
fig.tight_layout()
plt.show();

**Figure.** Generated MNIST digits after 20 epochs of training. The model generates recognizable digit shapes, showing that the reverse diffusion process has learned a useful generative distribution over MNIST.

<br>

We can also visualize the denoising trajectory for a single sample to see how the image emerges from noise:

In [ ]:
#| code-fold: true
_, traj = sample(
    model, schedule, n_samples=1, shape=(1, 28, 28),
    device=DEVICE, return_trajectory=True
)

labels = ["$\mathbf{x}_T$", "$\mathbf{x}_{800}$", "$\mathbf{x}_{600}$",
          "$\mathbf{x}_{400}$", "$\mathbf{x}_{200}$", "$\mathbf{x}_0$"]

fig, axes = plt.subplots(1, len(traj), figsize=(10, 2))
for ax, frame, lbl in zip(axes, traj, labels):
    ax.imshow(frame.squeeze().numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(lbl, fontsize=10)
    ax.axis("off")
fig.tight_layout()
plt.show();

**Figure.** Denoising trajectory from $\mathbf{x}_T$ (pure noise) to $\mathbf{x}_0$ (generated sample). Structure begins to emerge around $\mathbf{x}_{400}$ and sharpens in the final steps.

## Accelerated sampling: DDIM

The main drawback of DDPM is that generating a single sample requires $T = 1000$ forward passes through the neural network. Song et al. [@ddim] introduced **Denoising Diffusion Implicit Models** (DDIM), which reinterpret the generative process as a non-Markovian diffusion over a subsequence of timesteps while keeping the same marginals $q(\mathbf{x}_t | \mathbf{x}_0)$ as DDPM. This allows sampling with far fewer steps.

The key insight is that we never needed the Markov structure $q(\mathbf{x}_t | \mathbf{x}_{t-1})$ to train the model — only the marginals $q(\mathbf{x}_t | \mathbf{x}_0)$ appear in the training objective. DDIM defines a deterministic reverse process by first predicting $\hat{\mathbf{x}}_0$ from $\mathbf{x}_t$ using the model's noise prediction, then re-noising to the target level $t - S$:

$$
\mathbf{x}_{t-S} = \sqrt{\bar{\alpha}_{t-S}} \underbrace{\frac{\mathbf{x}_t - \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)}{\sqrt{\bar{\alpha}_t}}}_{\hat{\mathbf{x}}_0} + \sqrt{1 - \bar{\alpha}_{t-S}}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t).
$$

This update uses the [noise addition formula](#forward-process) in reverse: given a predicted clean image $\hat{\mathbf{x}}_0$ and the same noise direction $\boldsymbol{\epsilon}_\theta$, we re-corrupt to level $t - S.$ Because the process is deterministic ($\eta = 0$), the same starting noise always produces the same sample — a useful property for controlled generation. With stride $S = 20$, we reduce the number of steps from $1000$ to $50$.

:::{.callout-note}
DDIM uses the **same trained model** as DDPM — no retraining is needed. The speed-up comes entirely from the sampling procedure. The $\eta$ parameter in the full DDIM formulation interpolates between deterministic ($\eta = 0$) and stochastic ($\eta = 1$, recovering DDPM) sampling.

:::

Implementing the DDIM sampler with configurable stride:

In [ ]:
@torch.inference_mode()
def ddim_sample(
    model: nn.Module,
    schedule: DiffusionSchedule,
    n_samples: int,
    shape: tuple[int, ...],
    device: torch.device,
    stride: int = 20,
) -> torch.Tensor:
    """Deterministic DDIM sampler with configurable step stride."""
    model.eval()

    # Build the subsequence of timesteps [T-1, T-1-S, ..., 0]
    timesteps = list(range(schedule.T - 1, -1, -stride))          # <1>
    if timesteps[-1] != 0:
        timesteps.append(0)

    xt = torch.randn(n_samples, *shape, device=device)

    for i, t_idx in enumerate(timesteps):
        t_batch = torch.full((n_samples,), t_idx, device=device, dtype=torch.long)

        # Predict noise
        eps_pred = model(xt, t_batch)

        # Recover predicted x_0
        sqrt_acp_t = schedule.sqrt_acp[t_idx]
        sqrt_om_t  = schedule.sqrt_one_minus_acp[t_idx]
        x0_pred = (xt - sqrt_om_t * eps_pred) / sqrt_acp_t        # <2>
        x0_pred = x0_pred.clamp(-1, 1)                             # <3>

        # Re-noise to next (lower) timestep
        if i + 1 < len(timesteps):
            t_next = timesteps[i + 1]
            sqrt_acp_next = schedule.sqrt_acp[t_next]
            sqrt_om_next  = schedule.sqrt_one_minus_acp[t_next]
            xt = sqrt_acp_next * x0_pred + sqrt_om_next * eps_pred # <4>
        else:
            xt = x0_pred                                           # <5>

    return xt.cpu()

1. Construct a decreasing sequence of timestep indices taking every $S$-th step. With $T = 1000$ and $S = 20$ this gives 50 steps.
2. Recover the predicted clean image $\hat{\mathbf{x}}_0 = (\mathbf{x}_t - \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}_\theta) / \sqrt{\bar{\alpha}_t}$ from the one-shot formula.
3. Clamp $\hat{\mathbf{x}}_0$ to $[-1, 1]$ to prevent accumulation of numerical errors over many steps.
4. Re-corrupt $\hat{\mathbf{x}}_0$ to the next noise level $t_{\text{next}}$ using the same predicted noise direction.
5. At the final step, return the predicted clean image directly.

Comparing DDPM (1000 steps) and DDIM (50 steps) side by side:

In [ ]:
#| code-fold: true
torch.manual_seed(RANDOM_SEED)
samples_ddpm = sample(model, schedule, n_samples=20, shape=(1, 28, 28), device=DEVICE)

torch.manual_seed(RANDOM_SEED)
samples_ddim = ddim_sample(model, schedule, n_samples=20, shape=(1, 28, 28), device=DEVICE, stride=20)

fig, axes = plt.subplots(4, 10, figsize=(12, 5))

# Top two rows: DDPM
for i, ax in enumerate(axes[:2].flat):
    ax.imshow(samples_ddpm[i].squeeze().numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")

# Bottom two rows: DDIM
for i, ax in enumerate(axes[2:].flat):
    ax.imshow(samples_ddim[i].squeeze().numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")

# Row labels
axes[0, 0].set_ylabel("DDPM\n(1000 steps)", fontsize=8, rotation=90, labelpad=4)
axes[2, 0].set_ylabel("DDIM\n(50 steps)", fontsize=8, rotation=90, labelpad=4)

fig.tight_layout()
plt.show();

**Figure.** Top two rows: DDPM samples (1000 steps). Bottom two rows: DDIM samples (50 steps, stride 20) from the same trained model. DDIM produces comparable visual quality at $20\times$ the speed, demonstrating that the full Markov chain is not required for high-quality generation.

■